In [5]:
import pandas as pd
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from sklearn.model_selection import train_test_split

# 1. Connect to your NEW, verified project asset
ml_client = MLClient.from_config(credential=DefaultAzureCredential())
data_asset = ml_client.data.get(name="library_occupancy_final", version="1")

# 2. Load and Sample (Crucial to keep RAM under control)
df = pd.read_parquet(data_asset.path).sample(frac=0.2, random_state=42)

# 3. Sort by Time to ensure Scientific Rigor
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp')

# 4. Mandatory 4-Way Split
train_val_test, deploy_df = train_test_split(df, test_size=0.10, shuffle=False)
train_df, temp_df = train_test_split(train_val_test, test_size=0.333, shuffle=True, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, shuffle=True, random_state=42)

print(f"SUCCESS! Project data loaded.")
print(f"Train Rows: {len(train_df)} | Deploy Rows: {len(deploy_df)}")

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


SUCCESS! Project data loaded.
Train Rows: 2427144 | Deploy Rows: 404322


In [6]:
import numpy as np
from sklearn.metrics import mean_absolute_error

# Predict the average occupancy of the training set for every row in the test set
baseline_guess = train_df['meter_reading'].mean()
baseline_preds = np.full(shape=len(test_df), fill_value=baseline_guess)

# Calculate MAE
baseline_mae = mean_absolute_error(test_df['meter_reading'], baseline_preds)

print(f"Baseline Mean Prediction: {baseline_guess:.2f}")
print(f"Baseline MAE (The bar for your AI): {baseline_mae:.2f}")

Baseline Mean Prediction: 2255.41
Baseline MAE (The bar for your AI): 3822.78


In [7]:
from sklearn.ensemble import RandomForestRegressor
import mlflow.sklearn

In [8]:
# 1. Feature Engineering: Extract Time from the timestamp
train_df['hour'] = train_df['timestamp'].dt.hour
train_df['day_of_week'] = train_df['timestamp'].dt.dayofweek

test_df['hour'] = test_df['timestamp'].dt.hour
test_df['day_of_week'] = test_df['timestamp'].dt.dayofweek

# 2. Update Feature List (This is the magic fix)
# We add hour and day because library occupancy follows a strict schedule
features = ['air_temperature', 'cloud_coverage', 'dew_temperature', 'hour', 'day_of_week']

X_train = train_df[features].fillna(0)
y_train = train_df['meter_reading']

X_test = test_df[features].fillna(0)
y_test = test_df['meter_reading']

# 3. Retrain the model
from sklearn.ensemble import RandomForestRegressor
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# 4. Save/Register this as Version 3
import mlflow
with mlflow.start_run(run_name="Time_Feature_Improvement"):
    mlflow.sklearn.log_model(rf_model, "model")
    # Register this version
    run_id = mlflow.active_run().info.run_id
    model_uri = f"runs:/{run_id}/model"
    mlflow.register_model(model_uri, "library_occupancy_rf_model")

2026/04/10 22:41:47 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpddj9czfc/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.2.2', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/04/10 22:41:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'library_occupancy_rf_model' already exists. Creating a new version of this model...
2026/04/10 22:41:51 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: library_occupancy_rf_model, version 4
Created version '4' of model 'library_occupancy_rf_model'.
2026/04/10 22:41:51 INFO mlflow.tracking._tracking_service.client: 🏃 View run Time_Feature_Improvement at: https://qatarcentral.api.azureml.ms/mlflow/v2.0/

In [9]:
import mlflow
import pandas as pd

# --- DYNAMIC VERSION FIX ---
model_name = "library_occupancy_rf_model"
# This line automatically finds the highest version number in your Azure registry
latest_version = max([int(m.version) for m in ml_client.models.list(name=model_name)])
print(f"Loading latest model: Version {latest_version}")

model_uri = f"models:/{model_name}/{latest_version}"
loaded_model = mlflow.pyfunc.load_model(model_uri)
# ---------------------------

# The rest of your code remains the same
features = ['air_temperature', 'cloud_coverage', 'dew_temperature', 'hour', 'day_of_week']
X_test_sample = test_df[features].fillna(0).head(10)
y_actual_sample = test_df['meter_reading'].head(10).values

predictions = loaded_model.predict(X_test_sample)

results_table = pd.DataFrame({
    'Actual Occupancy': y_actual_sample,
    f'Model Prediction (v{latest_version})': predictions
})
display(results_table)

,Actual Occupancy,Model Prediction (v4)
0,173.000,2612.318308
1,7587.890,2737.954316
2,494.500,1771.914922
3,175.948,1583.381294
4,0.000,2352.418716
5,5.010,1758.427280
6,0.000,884.521942
7,309.073,2323.032252
8,121.281,1794.251722
9,1742.190,1619.991135
